# 인사 데이터 JSONL 변환

### 라이브러리 선언

In [1]:
import pandas as pd
import json
from pathlib import Path
from datetime import datetime
import os
from dotenv import load_dotenv

print(f'pandas 버전: {pd.__version__}')
print('라이브러리 로딩 완료!')

pandas 버전: 3.0.2
라이브러리 로딩 완료!


### 환경 설정 *** 경로 확인 필요

In [2]:
# .env 파일의 설정값을 환경변수로 불러옴
load_dotenv()

# 전처리된 CSV 파일이 있는 폴더 경로
INPUT_DIR  = Path(os.getenv('INPUT_DIR',  '../Preprocessing/output'))
# 변환된 JSONL 파일을 저장할 폴더 경로
OUTPUT_DIR = Path(os.getenv('OUTPUT_DIR', 'output'))

# 출력 폴더가 없으면 자동으로 생성
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f'  입력 디렉토리: {INPUT_DIR}')
print(f'  출력 디렉토리: {OUTPUT_DIR}')

  입력 디렉토리: ..\Preprocessing\output
  출력 디렉토리: output


# 1. 데이터 로딩

In [3]:
csv_files = sorted(INPUT_DIR.glob('*.csv'))

if not csv_files:
    print(f'CSV 파일 없음: {INPUT_DIR}')
    print('전처리 노트북을 먼저 실행해 주세요.')
else:
    dfs = {}
    for path in csv_files:
        df = pd.read_csv(path, encoding='utf-8-sig', dtype=object)
        dfs[path.stem] = df
        print(f'  로딩: {path.name}  ({len(df):,}행 / {len(df.columns)}열)')

    basic_df = dfs.get('기본인사정보_정제', pd.DataFrame())

    basic_lookup = {}
    if not basic_df.empty:
        for row in basic_df.to_dict('records'):
            emp_id = str(row.get('사원번호', '')).strip()
            basic_lookup[emp_id] = {
                '이름':     str(row.get('이름', '')),
                '부서':     str(row.get('부서', '')),
                '부서레벨': str(row.get('부서레벨', '')),
                '직급':     str(row.get('직급', '')),
                '직급레벨': str(row.get('직급레벨', '')),
            }

    print(f'\n로딩 완료! 총 {len(dfs)}개 파일')
    print(f'기본인사정보 조회 딕셔너리: {len(basic_lookup):,}건')

  로딩: 급여정보_정제.csv  (2,000행 / 7열)
  로딩: 기본인사정보_정제.csv  (2,000행 / 30열)
  로딩: 역량성과_정제.csv  (2,000행 / 13열)


  로딩: 통합인사정보_정제.csv  (2,000행 / 48열)

로딩 완료! 총 4개 파일
기본인사정보 조회 딕셔너리: 2,000건


# 2. 변환 함수 정의

In [4]:
MISSING_VALUES = {'', '미입력', 'nan', 'NaN', 'None', 'none'}

EMBEDDING_FIELDS = [
    # 기본인사정보
    '성별', '나이', '생년월일', '주민등록번호', '병역',
    '입사일', '근속기간',
    '학력', '출신대학', '학점', '채용경로', '계약형태',
    '이전직장명', '이전최종직급', '이전담당업무',
    '회사명', '사업장위치',
    '팀', '직책',
    '퇴직구분', '퇴직일자',
    '이메일', '전화번호', '주소',
    # 급여정보
    '연봉', '잔업시간', '미사용휴가일수', '급여은행', '계좌번호', '4대보험가입여부',
    # 역량성과
    '성과점수',
    '인사고과_2020', '인사고과_2021', '인사고과_2022', '인사고과_2023', '인사고과_2024',
    '자격증', 'TOEIC점수', '자격증수당여부', '포상이력', '징계이력', '징계사유',
]


def clean(val):
    cleaned = str(val).strip()
    return cleaned if cleaned not in MISSING_VALUES else ''


def build_embedding_text(row, info):
    fields = [
        ('이름', info.get('이름', clean(row.get('이름', '')))),
        ('부서', info.get('부서', clean(row.get('부서', '')))),
        ('직급', info.get('직급', clean(row.get('직급', '')))),
    ]
    for field in EMBEDDING_FIELDS:
        val = clean(str(row.get(field, '')))
        if val:
            fields.append((field, val))

    seen_keys = []
    parts = []
    for key, val in fields:
        if val and key not in seen_keys:
            parts.append(f'{key}: {val}')
            seen_keys.append(key)

    return ' '.join(parts)


def to_record(row, source_name):
    emp_id = str(row.get('사원번호', '')).strip()
    info   = basic_lookup.get(emp_id, {})

    return {
        'employee_id':      emp_id,
        'employee_name':    info.get('이름', clean(row.get('이름', ''))),
        'department':       info.get('부서', clean(row.get('부서', ''))),
        'department_level': clean(row.get('부서레벨', '')) or info.get('부서레벨', ''),
        'position':         info.get('직급', clean(row.get('직급', ''))),
        'position_level':   clean(row.get('직급레벨', '')) or info.get('직급레벨', ''),
        'embedding_text':   build_embedding_text(row, info),
        'source':           source_name.replace('_정제', ''),
        'timestamp':        datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        'changed':          [],
        'embedding_vector': [],
    }


print('변환 함수 정의 완료!')

변환 함수 정의 완료!


# 3. JSONL 변환 및 저장

In [5]:
saved_files = []

for source_name, df in dfs.items():
    out_path = OUTPUT_DIR / f'{source_name}.jsonl'

    old_records = {}
    if out_path.exists():
        with open(out_path, 'r', encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                rec = json.loads(line)
                old_records[rec.get('employee_id', '')] = rec

    rows = df.to_dict('records')

    with open(out_path, 'w', encoding='utf-8') as f:
        for row in rows:
            record = to_record(row, source_name)
            emp_id = record['employee_id']

            if emp_id in old_records:
                old = old_records[emp_id]

                record['changed'] = old.get('changed', [])

                skip = {'embedding_vector', 'embedding_text', 'timestamp', 'changed'}
                new_fields = []
                for field, new_val in record.items():
                    if field in skip:
                        continue
                    old_val = old.get(field, '')
                    if str(old_val) != str(new_val):
                        new_fields.append({
                            'field': field,
                            'old':   str(old_val),
                            'new':   str(new_val),
                        })

                if new_fields:
                    record['changed'].append({
                        'timestamp': record['timestamp'],
                        'fields':    new_fields,
                    })

            f.write(json.dumps(record, ensure_ascii=False) + '\n')

    saved_files.append((out_path, len(rows)))
    print(f'  저장: {out_path.name}  ({len(rows):,}건)')

  저장: 급여정보_정제.jsonl  (2,000건)


  저장: 기본인사정보_정제.jsonl  (2,000건)


  저장: 역량성과_정제.jsonl  (2,000건)


  저장: 통합인사정보_정제.jsonl  (2,000건)


# 4. 저장 결과 확인

In [6]:
for out_path, _ in saved_files:
    # 저장된 JSONL 파일을 읽기 모드로 열기
    with open(out_path, 'r', encoding='utf-8') as f:
        # readline(): 파일의 첫 번째 줄(=첫 번째 레코드)만 읽음
        # json.loads(): JSON 문자열을 파이썬 딕셔너리로 변환
        sample = json.loads(f.readline())
    print(f'\n[{out_path.name}] 첫 번째 레코드:')
    # indent=2: 들여쓰기 2칸으로 보기 좋게 출력
    print(json.dumps(sample, ensure_ascii=False, indent=2))
    print()


[급여정보_정제.jsonl] 첫 번째 레코드:
{
  "employee_id": "EMP0001",
  "employee_name": "지준서",
  "department": "마케팅부",
  "department_level": "1",
  "position": "사원",
  "position_level": "1",
  "embedding_text": "이름: 지준서 부서: 마케팅부 직급: 사원 연봉: 32000000 잔업시간: 10 미사용휴가일수: 14 급여은행: 우리은행 계좌번호: 1002-324-124157 4대보험가입여부: 가입",
  "source": "급여정보",
  "timestamp": "2026-06-07 13:24:39",
  "changed": [],
  "embedding_vector": []
}


[기본인사정보_정제.jsonl] 첫 번째 레코드:
{
  "employee_id": "EMP0001",
  "employee_name": "지준서",
  "department": "마케팅부",
  "department_level": "1",
  "position": "사원",
  "position_level": "1",
  "embedding_text": "이름: 지준서 부서: 마케팅부 직급: 사원 성별: 남 나이: 30 생년월일: 1997-09-19 주민등록번호: 970919-1367382 병역: 군필 입사일: 2024-05-02 근속기간: 2 학력: 대졸 출신대학: 연세대학교 학점: 3.7 채용경로: 추천 계약형태: 정규직 회사명: 두리안정보기술 사업장위치: 서울 강남 팀: 브랜드팀 직책: 팀원 이메일: jijunseo16@nate.com 전화번호: 010-1647-2192 주소: 서울시 마포구 합정동",
  "source": "기본인사정보",
  "timestamp": "2026-06-07 13:24:39",
  "changed": [],
  "embedding_vector": []
}


[역량성과_정제.jsonl] 첫 번째 레코드: